## Local Inference on GPU 
Model page: https://huggingface.co/MahmoodLab/CONCH

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/MahmoodLab/CONCH)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [1]:
!GIT_TERMINAL_PROMPT=0 pip install -q "git+https://github.com/mahmoodlab/CONCH.git"

from kaggle_secrets import UserSecretsClient
from conch.open_clip_custom import create_model_from_pretrained
import torch

token = UserSecretsClient().get_secret("HF_TOKEN")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = create_model_from_pretrained("conch_ViT-B-16", "hf_hub:MahmoodLab/conch", hf_auth_token=token)
model = model.to(device).eval()
print("CONCH loaded on", device)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incom

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


meta.yaml:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

CONCH loaded on cuda


In [2]:
import numpy as np, glob
# locate Part 1 arrays
paths = {p.split("/")[-1]: p for p in glob.glob("/kaggle/input/**/Part 1/**/*.npy", recursive=True)}
IMAGES_PATH = paths["images.npy"]; MASKS_PATH = paths["masks.npy"]; TYPES_PATH = paths["types.npy"]
print("found:", IMAGES_PATH, MASKS_PATH, TYPES_PATH, sep="\n")

CLASS_NAMES = ["Neoplastic", "Inflammatory", "Connective", "Dead", "Epithelial"]
CROP = 64

def nuclei_in_image(mask):
    out = []
    for c in range(5):
        ch = mask[:, :, c]
        for inst in np.unique(ch):
            if inst == 0: continue
            ys, xs = np.where(ch == inst)
            out.append((c, int(inst), int(ys.mean()), int(xs.mean())))
    return out

def crop_patch(img, r, c, size=CROP):
    h = size // 2; R, C = img.shape[:2]
    r0, r1, c0, c1 = r-h, r+h, c-h, c+h
    pt, pl = max(0,-r0), max(0,-c0); pb, pr = max(0,r1-R), max(0,c1-C)
    r0, r1, c0, c1 = max(0,r0), min(R,r1), max(0,c0), min(C,c1)
    patch = img[r0:r1, c0:c1]
    patch = np.pad(patch, ((pt,pb),(pl,pr),(0,0)), mode="constant")
    return patch[:size, :size]

found:
/kaggle/input/datasets/andrewmvd/cancer-inst-segmentation-and-classification/Part 1/Images/images.npy
/kaggle/input/datasets/andrewmvd/cancer-inst-segmentation-and-classification/Part 1/Masks/masks.npy
/kaggle/input/datasets/andrewmvd/cancer-inst-segmentation-and-classification/Part 1/Images/types.npy


In [3]:
import numpy as np, torch
from PIL import Image
from tqdm import tqdm

images = np.load(IMAGES_PATH, mmap_mode="r")
masks  = np.load(MASKS_PATH,  mmap_mode="r")
types  = np.load(TYPES_PATH,  allow_pickle=True)
MIN_VALID, BATCH = 0.6, 256

crops, labels, tissues, img_idx = [], [], [], []
for i in tqdm(range(len(images)), desc="scan"):
    img = np.asarray(images[i]).astype(np.uint8); m = np.asarray(masks[i])
    for cls, inst, r, c in nuclei_in_image(m):
        h = CROP//2
        vh = min(r+h,256)-max(r-h,0); vw = min(c+h,256)-max(c-h,0)
        if (vh*vw)/(CROP*CROP) < MIN_VALID: continue
        crops.append(crop_patch(img, r, c)); labels.append(cls)
        tissues.append(str(types[i])); img_idx.append(i)
labels, tissues, img_idx = np.array(labels), np.array(tissues), np.array(img_idx)
print(f"{len(crops)} nuclei. counts: {np.bincount(labels)}")

emb_probe, emb_contrast = [], []
for s in tqdm(range(0, len(crops), BATCH), desc="embed"):
    t = torch.stack([preprocess(Image.fromarray(p)) for p in crops[s:s+BATCH]]).to(device)
    with torch.inference_mode():
        emb_probe.append(model.encode_image(t, proj_contrast=False, normalize=False).cpu().numpy())
        emb_contrast.append(model.encode_image(t, proj_contrast=True, normalize=True).cpu().numpy())
emb_probe = np.concatenate(emb_probe); emb_contrast = np.concatenate(emb_contrast)
print("probe:", emb_probe.shape, "contrast:", emb_contrast.shape)

np.savez_compressed("/kaggle/working/pannuke_fold1_conch.npz",
    emb_probe=emb_probe, emb_contrast=emb_contrast,
    labels=labels, tissues=tissues, img_idx=img_idx)
print("cached. NOW SAVE A VERSION so it persists.")

scan: 100%|██████████| 2656/2656 [01:50<00:00, 24.06it/s]


51365 nuclei. counts: [21276  8863 13368   790  7068]


embed: 100%|██████████| 201/201 [1:40:59<00:00, 30.15s/it]


probe: (51365, 512) contrast: (51365, 512)
cached. NOW SAVE A VERSION so it persists.


In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, balanced_accuracy_score

d = np.load("/kaggle/working/pannuke_fold1_conch.npz", allow_pickle=True)
X, y, img_idx = d["emb_probe"], d["labels"], d["img_idx"]

# group-aware split: whole images go to train OR test
rng = np.random.RandomState(0); imgs = np.unique(img_idx); rng.shuffle(imgs)
train_imgs = set(imgs[:int(0.75*len(imgs))].tolist())
tr = np.array([i in train_imgs for i in img_idx])

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
clf.fit(X[tr], y[tr]); pred = clf.predict(X[~tr])
print("Image-grouped balanced accuracy:", round(balanced_accuracy_score(y[~tr], pred), 3))
print(classification_report(y[~tr], pred, target_names=CLASS_NAMES, digits=3))

Image-grouped balanced accuracy: 0.733
              precision    recall  f1-score   support

  Neoplastic      0.841     0.758     0.797      5504
Inflammatory      0.654     0.740     0.694      2023
  Connective      0.703     0.667     0.684      3242
        Dead      0.428     0.812     0.561       191
  Epithelial      0.595     0.689     0.638      1723

    accuracy                          0.723     12683
   macro avg      0.644     0.733     0.675     12683
weighted avg      0.736     0.723     0.727     12683

